<a href="https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Relatividad/Relatividad_FLRW_Expansion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Investigación Avanzada: Relatividad FLRW Expansion

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

def friedmann_eq(t, a, H0, Omega_m, Omega_r, Omega_L, Omega_k):
    """
    Friedmann equation: da/dt = a * H0 * sqrt(Omega_m a^-3 + Omega_r a^-4 + Omega_L + Omega_k a^-2)
    a is the scale factor
    t is time in units of 1/H0 (Hubble time)
    """
    # Guard against a <= 0
    if a <= 0:
        return 0
    # The term inside the square root
    density = Omega_m * a**-3 + Omega_r * a**-4 + Omega_L + Omega_k * a**-2
    if density < 0:
        return 0
    dadt = a * np.sqrt(density)
    return dadt

def simulate_flrw():
    # Cosmological parameters
    H0 = 1.0 # Work in units of Hubble time (approx 14.4 Gyr)
    
    # 1. Matter-dominated universe
    params_m = (1.0, 0.0, 0.0, 0.0)
    # 2. Radiation-dominated universe
    params_r = (0.0, 1.0, 0.0, 0.0)
    # 3. Lambda-dominated universe
    params_L = (0.0, 0.0, 1.0, 0.0)
    # 4. LCDM (concordance model)
    params_lcdm = (0.315, 9e-5, 0.685 - 9e-5, 0.0)
    
    t_span = (1e-5, 2.0)
    a0 = [1e-5] # start close to big bang
    
    models = {
        'Matter only (EdS)': params_m,
        'Radiation only': params_r,
        'Dark Energy only (de Sitter)': params_L,
        'Lambda-CDM': params_lcdm
    }
    
    plt.figure(figsize=(10, 6))
    
    for name, params in models.items():
        if name == 'Dark Energy only (de Sitter)':
            # Analytical for pure Lambda (a(t) ~ exp(H0 t))
            t_eval = np.linspace(0, 2, 100)
            a_eval = 1e-3 * np.exp(t_eval) # arbitrary normalization
            plt.plot(t_eval, a_eval, '--', label=name)
        else:
            sol = solve_ivp(friedmann_eq, t_span, a0, args=(H0, *params), 
                            dense_output=True, method='RK45', rtol=1e-8)
            plt.plot(sol.t, sol.y[0], label=name)
            
            # Find present day (a=1)
            # Find index where a is closest to 1
            idx = np.argmin(np.abs(sol.y[0] - 1.0))
            t_today = sol.t[idx]
            plt.plot(t_today, 1.0, 'ko')

    plt.axhline(1.0, color='gray', linestyle=':', label='Present Day (a=1)')
    plt.xlabel('Time (in units of $H_0^{-1}$ $\approx 14.4$ Gyr)')
    plt.ylabel('Scale Factor $a(t)$')
    plt.title('FLRW Universe Expansion for Different Cosmological Models')
    plt.ylim(0, 3)
    plt.xlim(0, 2)
    plt.legend()
    plt.grid(True)
    plt.savefig('FLRW_Expansion.png', dpi=300)
    print("FLRW simulation complete. Image saved.")

if __name__ == '__main__':
    simulate_flrw()
